# VoxCPM2 — Indian female voice test (Colab GPU)

Open-source voice for the Galaxy Toyota / Hans Hyundai / Auto Car Repair avatar.

1. **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
2. Cell 3 makes **6 new (fictional) voices** — no real person's recording is used, so no consent is needed.
3. Listen, pick the best young Indian female voice, set `CHOSEN` in cell 4 and hear it in Hindi, Hinglish and English.
4. Download `chosen_voice.wav` + `chosen_voice.txt`: the same voice is used later on the GPU server.

Copying a real person's voice needs their signed consent (`voice_clone/consent/`); this notebook does not do that.

In [ ]:
#@title 1) Install  (if pip asks to restart: Runtime → Restart session, then run cell 2)
!pip install -q -U voxcpm soundfile

In [ ]:
#@title 2) Load VoxCPM2 on the GPU
import inspect, time, os
import numpy as np, torch, torch._dynamo
torch._dynamo.config.suppress_errors = True
torch._dynamo.config.disable = True            # eager mode: avoids torch.compile threading bugs
from voxcpm import VoxCPM
from IPython.display import Audio, display
import soundfile as sf

assert torch.cuda.is_available(), "No GPU: Runtime → Change runtime type → T4 GPU"
print("GPU:", torch.cuda.get_device_name(0))
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=True, optimize=False)
SAMPLE_RATE = model.tts_model.sample_rate
SIG = inspect.signature(model.generate).parameters
print("sample rate:", SAMPLE_RATE)
print("generate() parameters:", list(SIG))

def _kwargs(text, ref, ref_text, steps):
    kw = {"text": text}
    if "inference_timesteps" in SIG:
        kw["inference_timesteps"] = steps
    if ref:
        name = next((k for k in ("prompt_wav_path", "reference_wav_path", "prompt_audio_path") if k in SIG), None)
        if not name:
            raise RuntimeError(f"no reference-audio parameter in {list(SIG)}")
        kw[name] = ref
        tname = next((k for k in ("prompt_text", "reference_text") if k in SIG), None)
        if tname and ref_text is not None:
            kw[tname] = ref_text
    return kw

def say(text, ref=None, ref_text=None, steps=10, seed=None, show=True):
    """Speak `text` (optionally in the voice of the reference clip). Returns the waveform."""
    if seed is not None:
        torch.manual_seed(seed); np.random.seed(seed)
    t = time.time()
    out = model.generate(**_kwargs(text, ref, ref_text, steps))
    if isinstance(out, (np.ndarray, torch.Tensor)):
        parts = [out]
    else:                                         # streaming: a generator of chunks
        parts = list(out)
    wav = np.concatenate([np.asarray(p.detach().cpu() if hasattr(p, "detach") else p, dtype=np.float32).reshape(-1)
                          for p in parts])
    dt, dur = time.time() - t, len(wav) / SAMPLE_RATE
    print(f"{dt:.1f} s to make {dur:.1f} s of audio (real-time factor {dt / max(dur, 1e-6):.2f}; < 1 = faster than speech)")
    if show:
        display(Audio(wav, rate=SAMPLE_RATE))
    return wav

_ = say("Hello, warm up test.", steps=5, show=False)
print(f"ready — VRAM {torch.cuda.memory_allocated() / 1e9:.1f} GB")

In [ ]:
#@title 3) Six new fictional voices — listen and note the best young Indian female one
GREETING = "Namaste! Welcome to Galaxy Toyota. Which Toyota model would you like to know about today?"
os.makedirs("/content/voices", exist_ok=True)
for seed in range(6):
    print(f"--- voice {seed} ---")
    wav = say(GREETING, seed=seed)
    sf.write(f"/content/voices/voice_{seed}.wav", wav, SAMPLE_RATE)
print("Not good enough? Change range(6) to range(6, 12) and run again.")

In [ ]:
#@title 4) The chosen voice in Hindi, Hinglish and English
CHOSEN = 0   #@param {type:"integer"}
REF = f"/content/voices/voice_{CHOSEN}.wav"
REF_TEXT = GREETING
tests = {
    "Hindi":    "नमस्ते! मैं गैलेक्सी टोयोटा से बात कर रही हूँ। फॉर्च्यूनर की कीमत चौंतीस लाख से पचास लाख रुपये तक है।",
    "Hinglish": "Namaste! Main Galaxy Toyota se baat kar rahi hoon. Aap kis Toyota model ke baare mein jaanna chahenge?",
    "English":  "The Toyota Fortuner is a seven-seat SUV. Shall I show you its page, or book a test drive?",
}
for name, text in tests.items():
    print(f"--- {name} ---")
    say(text, ref=REF, ref_text=REF_TEXT)

In [ ]:
#@title 5) Speed with fewer steps (lower = faster, slightly lower quality)
for steps in (10, 6, 4):
    print(f"steps={steps}")
    say(tests["Hinglish"], ref=REF, ref_text=REF_TEXT, steps=steps)

In [ ]:
#@title 6) Download the chosen voice (used on the GPU server later)
import shutil
from google.colab import files
shutil.copy(REF, "/content/chosen_voice.wav")
open("/content/chosen_voice.txt", "w", encoding="utf-8").write(REF_TEXT)
files.download("/content/chosen_voice.wav")
files.download("/content/chosen_voice.txt")